In [ ]:
from typing import TypedDict, List, Literal, Annotated
from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
import arxiv
import os

In [ ]:
class ResearchState(TypedDict, total=False):
    messages: Annotated[List[BaseMessage], add_messages]
    context: List[str]
    research_topic: str
    search_results: List[dict]
    sources: List[dict]
    findings: List[str]
    iteration: int

In [ ]:
llm = ChatGroq(
        api_key= os.getenv('GROQ_API_KEY'),
        model="llama-3.1-8b-instant"
    )

In [ ]:
# Internet Search tool
search_tool = DuckDuckGoSearchRun(region="us-en")

# Calculator tool
@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}
        
        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    except Exception as e:
        return {"error": str(e)}


# ArXiv Paper Search tool
@tool
def arxiv_search(query: str, max_results: int = 5) -> list:
    """
    Search for papers on arXiv using a keyword query.
    Returns a list of paper titles and IDs.
    """
    try:
        search = arxiv.Search(
            query=query,
            max_results=max_results,
            sort_by=arxiv.SortCriterion.Relevance
        )
        
        papers = []
        for result in search.results():
            papers.append({
                "title": result.title,
                "id": result.get_short_id(),
                "url": result.entry_id,
                "authors": [author.name for author in result.authors],
                "abstract": result.summary[:200] + "..." if len(result.summary) > 200 else result.summary
            })
        
        return papers
    except Exception as e:
        return {"error": str(e)}

In [ ]:
def synthesize_findings(state: ResearchState):
    """Combine all findings into a coherent report"""
    findings_text = "\n".join(state.get("findings", []))
    
    messages = [
        SystemMessage(content="Synthesize research findings into a clear report. Include citations."),
        HumanMessage(content=f"Topic: {state['research_topic']}\n\nFindings:\n{findings_text}")
    ]
    response = llm.invoke(messages)
    
    return {"messages": [AIMessage(content=response.content)]}

In [ ]:
tools = [search_tool, calculator, arxiv_search]

llm_with_tools = llm.bind_tools(tools)

tool_node = ToolNode(tools)

In [ ]:
def chat_node(state: ResearchState):
    """LLM node that may answer or request tool calls"""    
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    
    return {"messages": [response]}

In [ ]:
graph = StateGraph(ResearchState)
 
# Add nodes
graph.add_node("chat", chat_node)
graph.add_node("tools", tool_node)
graph.add_node("synthesize", synthesize_findings)
 
# Entry point
graph.add_edge(START, "chat")
graph.add_conditional_edges("chat", tools_condition, {"tools": "tools", END: END, "synthesize": "synthesize"})
graph.add_edge("tools", "chat")
graph.add_edge("synthesize", END)
 
# Compile
workflow = graph.compile()
workflow

In [60]:
state = {
    "messages": [],
    "context": [],
    "research_topic": "",
    "search_results": [],
    "sources": [],
    "findings": [],
    "iteration": 0
}

while True:
    user_input = input("You: ")

    if user_input in ["exit", "quit"]:
        break

    state["messages"].append(HumanMessage(content=user_input))
    state = workflow.invoke(state)

    print("Assistant:", state["messages"][-1].content)

Assistant: Based on the search results, the current temperature in Mumbai is around 32°C.
Assistant: It was nice chatting with you, Arya. Is there anything else I can help you with?


/tmp/ipykernel_7070/2814858204.py:45: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


Assistant: Based on the search results, some new research in the field of artificial intelligence includes:

1. The Artificial Scientist: Logicist, Emergentist, and Universalist Approaches to Artificial General Intelligence, which explores the necessary conditions for constructing an Artificial Scientist and evaluates several approaches to artificial general intelligence (AGI).
2. Compression, The Fermi Paradox and Artificial Super-Intelligence, which discusses possible difficulties in communication with and control of an AGI.
3. Creative Problem Solving in Artificially Intelligent Agents: A Survey and Framework, which focuses on methods for solving off-nominal, or anomalous problems in autonomous systems.
4. A Review on Explainable Artificial Intelligence for Healthcare: Why, How, and When?, which reviews the applications of explainable AI in healthcare and discusses the importance of explainability in AI models.
5. Artificial Intelligence Framework for Simulating Clinical Decision-Ma